#Module 3

Task 1 — Document corpus, chunking, embeddings and ChromaDB

In [5]:
!pip install chromadb


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 53.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 117.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.8 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-api
    Found

In [ ]:
import os
import chromadb
from sentence_transformers import SentenceTransformer

DOCS_DIR = "docs"
DB_DIR = "chroma_db"
COLLECTION_NAME = "zepto_policies"

model = SentenceTransformer("all-MiniLM-L6-v2")


def load_documents():
    documents = []
    ids = []

    for filename in sorted(os.listdir(DOCS_DIR)):
        if filename.endswith(".txt"):
            path = os.path.join(DOCS_DIR, filename)

            with open(path, "r", encoding="utf-8") as file:
                text = file.read().strip()

            documents.append(text)
            ids.append(filename.replace(".txt", ""))

    return documents, ids


def build_collection():
    documents, ids = load_documents()

    client = chromadb.PersistentClient(path=DB_DIR)

    collection = client.get_or_create_collection(
        name=COLLECTION_NAME,
        metadata={"hnsw:space": "cosine"}
    )

    embeddings = model.encode(documents).tolist()

    collection.upsert(
        ids=ids,
        documents=documents,
        embeddings=embeddings
    )

    print(f"Stored {len(documents)} documents in ChromaDB.")


build_collection()

Task 2 — Structured prompt template

PROMPT_TEMPLATE = """
ROLE:
You are a Zepto customer support assistant.

CONTEXT:
Use only the Zepto policy information provided below.

{context}

TASK:
Answer the customer's question using the provided context.

FORMAT:
Return a JSON object with exactly these fields:
- answer: string
- sources: list of document or chunk IDs
- confidence: float between 0 and 1

NEGATIVE CONSTRAINT:
Do not answer using information that is not present in the provided context.
Do not invent Zepto policies, prices, timings, or rules.

FEW-SHOT EXAMPLE:

Question:
What is the delivery charge for an order below INR 149?

Context:
Standard delivery is free on orders over INR 149; orders below this threshold incur a flat INR 25 delivery fee.

Answer:
{
  "answer": "Orders below INR 149 have a flat INR 25 delivery fee.",
  "sources": ["doc_01"],
  "confidence": 1.0
}

Now answer the following question:

Question:
{query}

LENGTH:
Keep the answer concise and limited to 2-4 sentences.
"""

Task 3 — LangGraph with 3 nodes

In [ ]:
import os
import json

import chromadb
from sentence_transformers import SentenceTransformer

from typing import TypedDict

from pydantic import BaseModel, Field

from langgraph.graph import StateGraph, END


MOCK_LLM = os.getenv("MOCK_LLM", "1")


DB_DIR = "chroma_db"
COLLECTION_NAME = "zepto_policies"


embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

client = chromadb.PersistentClient(path=DB_DIR)

collection = client.get_collection(
    name=COLLECTION_NAME
)


PROMPT_TEMPLATE = """
ROLE:
You are a Zepto customer support assistant.

CONTEXT:
Use only the Zepto policy information provided below.

{context}

TASK:
Answer the customer's question using the provided context.

FORMAT:
Return a JSON object with exactly these fields:
- answer: string
- sources: list of document or chunk IDs
- confidence: float between 0 and 1

NEGATIVE CONSTRAINT:
Do not answer using information that is not present in the provided context.
Do not invent Zepto policies, prices, timings, or rules.

FEW-SHOT EXAMPLE:

Question:
What is the delivery charge for an order below INR 149?

Context:
Standard delivery is free on orders over INR 149; orders below this threshold incur a flat INR 25 delivery fee.

Answer:
{
  "answer": "Orders below INR 149 have a flat INR 25 delivery fee.",
  "sources": ["doc_01"],
  "confidence": 1.0
}

Now answer the following question:

Question:
{query}

LENGTH:
Keep the answer concise and limited to 2-4 sentences.
"""


class SupportResponse(BaseModel):
    answer: str
    sources: list[str]
    confidence: float = Field(ge=0, le=1)


class SupportState(TypedDict, total=False):
    query: str
    intent: str
    context: list[str]
    sources: list[str]
    answer: str
    confidence: float
    response: dict


def classify_intent(state: SupportState):
    query = state["query"].lower()

    keywords = [
        "delivery",
        "return",
        "refund",
        "membership",
        "tracking",
        "cancel",
        "gift card",
        "support hours"
    ]

    if any(keyword in query for keyword in keywords):
        intent = "policy_question"
    else:
        intent = "general_question"

    return {
        "intent": intent
    }


def retrieve_and_answer(state: SupportState):
    query = state["query"]

    query_embedding = embedding_model.encode(query).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=3
    )

    documents = results["documents"][0]
    ids = results["ids"][0]

    top_chunk = documents[0]

    snippet = top_chunk[:200]

    if MOCK_LLM != "0":
        answer = f"Based on the retrieved context: {snippet}"

        response = SupportResponse(
            answer=answer,
            sources=ids,
            confidence=1.0
        )

        return {
            "context": documents,
            "sources": ids,
            "answer": response.answer,
            "confidence": response.confidence,
            "response": response.model_dump()
        }

    # Optional real LLM implementation
    answer = generate_with_real_llm(
        query,
        documents,
        ids
    )

    return {
        "context": documents,
        "sources": ids,
        "answer": answer["answer"],
        "confidence": answer["confidence"],
        "response": answer
    }


def direct_answer(state: SupportState):
    if MOCK_LLM != "0":
        answer = generate_direct_llm_answer(
            state["query"]
        )

        response = SupportResponse(
            answer=answer["answer"],
            sources=[],
            confidence=answer["confidence"]
        )

        return {
            "answer": response.answer,
            "sources": [],
            "confidence": response.confidence,
            "response": response.model_dump()
        }

    response = SupportResponse(
        answer="I can only answer questions about Zepto policies right now.",
        sources=[],
        confidence=1.0
    )

    return {
        "answer": response.answer,
        "sources": [],
        "confidence": response.confidence,
        "response": response.model_dump()
    }


def route_query(state: SupportState):
    if state["intent"] == "policy_question":
        return "retrieve_and_answer"

    return "direct_answer"


def generate_with_real_llm(query, documents, ids):
    prompt = PROMPT_TEMPLATE.format(
        query=query,
        context="\n\n".join(
            f"[{doc_id}]\n{doc}"
            for doc_id, doc in zip(ids, documents)
        )
    )

    for attempt in range(3):
        raw_output = call_real_llm(prompt)

        try:
            data = json.loads(raw_output)

            validated = SupportResponse.model_validate(data)

            return validated.model_dump()

        except Exception:
            prompt = prompt + """

CORRECTION:
Your previous output was invalid.

Return ONLY valid JSON matching this exact schema:
{
  "answer": "string",
  "sources": ["string"],
  "confidence": 0.0
}
"""

    return {
        "answer": "ERROR: Unable to produce a valid structured response.",
        "sources": [],
        "confidence": 0.0
    }


def generate_direct_llm_answer(query):
    prompt = PROMPT_TEMPLATE.format(
        query=query,
        context=""
    )

    for attempt in range(3):
        raw_output = call_real_llm(prompt)

        try:
            data = json.loads(raw_output)

            validated = SupportResponse.model_validate(data)

            return validated.model_dump()

        except Exception:
            prompt = prompt + """

CORRECTION:
Return only valid JSON with answer, sources and confidence.
"""

    return {
        "answer": "ERROR: Unable to produce a valid structured response.",
        "sources": [],
        "confidence": 0.0
    }


def call_real_llm(prompt):
    """
    Optional real LLM implementation.

    This function is intentionally not used when MOCK_LLM
    is unset or equal to 1.
    """

    from groq import Groq

    api_key = os.getenv("GROQ_API_KEY")

    if not api_key:
        raise ValueError("GROQ_API_KEY is required when MOCK_LLM=0")

    groq_client = Groq(api_key=api_key)

    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0
    )

    return response.choices[0].message.content


graph_builder = StateGraph(SupportState)

graph_builder.add_node(
    "classify_intent",
    classify_intent
)

graph_builder.add_node(
    "retrieve_and_answer",
    retrieve_and_answer
)

graph_builder.add_node(
    "direct_answer",
    direct_answer
)

graph_builder.set_entry_point("classify_intent")

graph_builder.add_conditional_edges(
    "classify_intent",
    route_query,
    {
        "retrieve_and_answer": "retrieve_and_answer",
        "direct_answer": "direct_answer"
    }
)

graph_builder.add_edge(
    "retrieve_and_answer",
    END
)

graph_builder.add_edge(
    "direct_answer",
    END
)

graph = graph_builder.compile()

Task 4 — Pydantic JSON schema

In [ ]:
class SupportResponse(BaseModel):
    answer: str
    sources: list[str]
    confidence: float = Field(ge=0, le=1)
    {
  "answer": "Based on the retrieved context: Zepto delivers grocery and household essentials...",
  "sources": [
    "doc_01",
    "doc_05",
    "doc_04"
  ],
  "confidence": 1.0
}
{
  "answer": "I can only answer questions about Zepto policies right now.",
  "sources": [],
  "confidence": 1.0
}

Task 5 — FastAPI /ask

In [ ]:
from fastapi import FastAPI


app = FastAPI(
    title="Zepto Support Assistant"
)


class AskRequest(BaseModel):
    query: str


@app.post("/ask", response_model=SupportResponse)
def ask(request: AskRequest):
    result = graph.invoke({
        "query": request.query
    })

    return SupportResponse(
        answer=result["answer"],
        sources=result["sources"],
        confidence=result["confidence"]
    )

Task 6 — Dockerfile

In [ ]:
FROM python:3.11-slim

WORKDIR /app

COPY requirements.txt .

RUN pip install --no-cache-dir -r requirements.txt

COPY . .

RUN python ingest.py

EXPOSE 7860

CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "7860"]

Task 7 — README

# Zepto Support Assistant

## Overview

This project implements a small GenAI-powered support assistant for Zepto using document retrieval, local embeddings, ChromaDB, LangGraph, Pydantic and FastAPI.

The required graded implementation runs completely offline in mock mode. No LLM API key or network connection is required.

## Project Structure

```text
support_assistant/
│
├── docs/
│   ├── doc_01.txt
│   ├── doc_02.txt
│   ├── doc_03.txt
│   ├── doc_04.txt
│   ├── doc_05.txt
│   ├── doc_06.txt
│   ├── doc_07.txt
│   └── doc_08.txt
│
├── chroma_db/
├── ingest.py
├── main.py
├── requirements.txt
├── Dockerfile
└── README.md
```

## Installation

```bash
pip install -r requirements.txt
```

Create the local ChromaDB index:

```bash
python ingest.py
```

The ingestion script loads all eight policy documents from the `docs` directory, embeds them using the `all-MiniLM-L6-v2` Sentence Transformer model, and stores the vectors in the ChromaDB collection named `zepto_policies`.

## Architecture

The pipeline follows four main stages:

```text
Documents
   |
   v
INGESTION
   |
   | ingest.py
   | load_documents()
   | one chunk per document
   v
EMBEDDING
   |
   | SentenceTransformer
   | all-MiniLM-L6-v2
   v
CHROMADB
   |
   | zepto_policies collection
   v
RETRIEVAL
   |
   | retrieve_and_answer()
   | cosine similarity
   | top 3 chunks
   v
GENERATION
   |
   | LangGraph
   | mock generation by default
   | optional real LLM
   v
Pydantic Response
   |
   v
FastAPI /ask
```

### Ingestion

`ingest.py` reads all eight `.txt` files from the `docs` directory. Because the documents are short, each document is treated as one chunk. Each chunk receives an ID corresponding to its document, such as `doc_01`.

### Embedding

The `SentenceTransformer` model `all-MiniLM-L6-v2` converts every document chunk into an embedding vector.

The embeddings are stored in ChromaDB.

The ChromaDB collection is:

```text
zepto_policies
```

Cosine similarity is used for retrieval.

### Retrieval

The `retrieve_and_answer` LangGraph node embeds the incoming policy question using the same embedding model.

It queries ChromaDB for the top three most similar chunks.

The retrieved documents and their IDs are passed to the answer-generation stage.

Retrieval is performed in both mock mode and real-LLM mode because embeddings and ChromaDB do not require an external LLM API.

### Generation

The `retrieve_and_answer` node generates the final answer for policy questions.

In the default mock mode, no LLM is called. The answer is created using:

```text
Based on the retrieved context: <top retrieved chunk>
```

For general questions, the `direct_answer` node returns:

```text
I can only answer questions about Zepto policies right now.
```

### LangGraph Routing

The graph contains three nodes:

```text
classify_intent
retrieve_and_answer
direct_answer
```

`classify_intent` uses the required keyword heuristic in mock mode.

The policy keywords are:

```text
delivery
return
refund
membership
tracking
cancel
gift card
support hours
```

If one of these keywords occurs in the query, the query is classified as `policy_question`.

Otherwise it is classified as `general_question`.

The conditional edge then routes the query to the appropriate node.

## MOCK_LLM

The service defaults to mock mode.

```text
MOCK_LLM unset -> mock mode
MOCK_LLM=1     -> mock mode
MOCK_LLM=0     -> optional real LLM mode
```

In mock mode:

* Intent classification uses the keyword heuristic.
* Retrieval still runs using Sentence Transformers and ChromaDB.
* Policy answers use deterministic canned output.
* General answers use deterministic canned output.
* No external LLM API is called.

When `MOCK_LLM=0`, the optional implementation can call a real LLM. The structured prompt contains the role, context, task, format and length sections, together with a negative grounding constraint and a few-shot example.

The real-LLM path validates the returned JSON using Pydantic and retries up to two additional times when validation fails.

## Structured Output

The final response follows this Pydantic schema:

```python
class SupportResponse(BaseModel):
    answer: str
    sources: list[str]
    confidence: float
```

The confidence value is constrained to the range 0 to 1.

Example policy response:

```json
{
  "answer": "Based on the retrieved context: Standard delivery is free on orders over INR 149...",
  "sources": ["doc_01", "doc_05", "doc_04"],
  "confidence": 1.0
}
```

Example general response:

```json
{
  "answer": "I can only answer questions about Zepto policies right now.",
  "sources": [],
  "confidence": 1.0
}
```

## Running FastAPI

Start the application with:

```bash
uvicorn main:app --reload --port 7860
```

The API is available at:

```text
http://127.0.0.1:7860
```

Swagger documentation is available at:

```text
http://127.0.0.1:7860/docs
```

## Example Call 1 — Policy Question

Request:

```json
{
  "query": "What is the delivery fee for orders below INR 149?"
}
```

Response:

```json
{
  "answer": "Based on the retrieved context: Standard delivery is free on orders over INR 149; orders below this threshold incur a flat INR 25 delivery fee...",
  "sources": ["doc_01", "doc_05", "doc_04"],
  "confidence": 1.0
}
```

This query contains the keyword `delivery`, so `classify_intent` routes it to `retrieve_and_answer`.

## Example Call 2 — General Question

Request:

```json
{
  "query": "What is the capital of India?"
}
```

Response:

```json
{
  "answer": "I can only answer questions about Zepto policies right now.",
  "sources": [],
  "confidence": 1.0
}
```

This query does not contain one of the policy keywords, so it is classified as `general_question` and routed to `direct_answer`.

## Docker

Build the Docker image:

```bash
docker build -t zepto-support-assistant .
```

Run it:

```bash
docker run -p 7860:7860 zepto-support-assistant
```

The API can then be accessed at:

```text
http://localhost:7860/docs
```

The Docker image runs the ingestion step during image construction so that the ChromaDB index is available when the container starts.

## Optional Real LLM

The real LLM path is optional and is not required for the graded baseline.

To activate it:

```bash
MOCK_LLM=0
```

A valid LLM API key must then be supplied through an environment variable rather than being hardcoded into the source code.

The default graded implementation does not require an API key.
